**EXTRACT LINKS**

In [1]:
"""
Bước 1: Crawl toàn bộ link nhân vật từ tất cả các trang
Site: https://baotanglichsu.vn/vi/Articles/3098/nhan-vat-lich-su/
Output: character_links.txt (mỗi dòng 1 URL đầy đủ)
"""

import requests
from bs4 import BeautifulSoup
import time
import re

BASE_URL = "https://baotanglichsu.vn"
LIST_URL = BASE_URL + "/vi/Articles/3098/nhan-vat-lich-su/Page{page}"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

OUTPUT_FILE = "character_links.txt"


def get_last_page(soup):
    """Đọc số trang cuối từ pagination '>>' """
    pagination = soup.select("ul.pagination li a")
    last_page = 1
    for a in pagination:
        href = a.get("href", "")
        # Link '>>' trỏ đến trang cuối: /vi/Articles/3098/.../PageXX
        if ">>" in a.text or "»" in a.text:
            match = re.search(r"Page(\d+)", href)
            if match:
                last_page = int(match.group(1))
    return last_page


def extract_character_links(soup):
    """Lấy tất cả link nhân vật trong trang danh sách"""
    links = []
    # Link nhân vật nằm trong h2 > a, bên trong col-md-9
    content_div = soup.select_one("div.col-md-9")
    if not content_div:
        return links
    for h2 in content_div.find_all("h2"):
        a = h2.find("a")
        if a and a.get("href"):
            href = a["href"]
            # Đảm bảo URL đầy đủ
            if href.startswith("/"):
                href = BASE_URL + href
            links.append(href)
    return links


def crawl_all_links():
    all_links = []

    # --- Lấy trang 1 để biết tổng số trang ---
    print("Đang lấy trang 1...")
    resp = requests.get(LIST_URL.format(page=1), headers=HEADERS, timeout=15)
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")

    last_page = get_last_page(soup)
    print(f"Tổng số trang: {last_page}")

    links = extract_character_links(soup)
    print(f"  Trang 1: {len(links)} nhân vật")
    all_links.extend(links)

    # --- Lặp qua các trang còn lại ---
    for page in range(2, last_page + 1):
        try:
            print(f"Đang lấy trang {page}/{last_page}...")
            resp = requests.get(LIST_URL.format(page=page), headers=HEADERS, timeout=15)
            resp.encoding = "utf-8"
            soup = BeautifulSoup(resp.text, "html.parser")
            links = extract_character_links(soup)
            print(f"  Trang {page}: {len(links)} nhân vật")
            all_links.extend(links)
            time.sleep(0.5)  # Nghỉ 0.5s giữa mỗi request để tránh bị chặn
        except Exception as e:
            print(f"  Lỗi trang {page}: {e}")

    # --- Loại trùng, giữ thứ tự ---
    seen = set()
    unique_links = []
    for link in all_links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)

    # --- Lưu ra file ---
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for link in unique_links:
            f.write(link + "\n")

    print(f"\n✅ Xong! Tổng cộng {len(unique_links)} link nhân vật.")
    print(f"   Đã lưu vào: {OUTPUT_FILE}")


if __name__ == "__main__":
    crawl_all_links()

Đang lấy trang 1...
Tổng số trang: 56
  Trang 1: 10 nhân vật
Đang lấy trang 2/56...
  Trang 2: 10 nhân vật
Đang lấy trang 3/56...
  Trang 3: 10 nhân vật
Đang lấy trang 4/56...
  Trang 4: 10 nhân vật
Đang lấy trang 5/56...
  Trang 5: 10 nhân vật
Đang lấy trang 6/56...
  Trang 6: 10 nhân vật
Đang lấy trang 7/56...
  Trang 7: 10 nhân vật
Đang lấy trang 8/56...
  Trang 8: 10 nhân vật
Đang lấy trang 9/56...
  Trang 9: 10 nhân vật
Đang lấy trang 10/56...
  Trang 10: 10 nhân vật
Đang lấy trang 11/56...
  Trang 11: 10 nhân vật
Đang lấy trang 12/56...
  Trang 12: 10 nhân vật
Đang lấy trang 13/56...
  Trang 13: 10 nhân vật
Đang lấy trang 14/56...
  Trang 14: 10 nhân vật
Đang lấy trang 15/56...
  Trang 15: 10 nhân vật
Đang lấy trang 16/56...
  Trang 16: 10 nhân vật
Đang lấy trang 17/56...
  Trang 17: 10 nhân vật
Đang lấy trang 18/56...
  Trang 18: 10 nhân vật
Đang lấy trang 19/56...
  Trang 19: 10 nhân vật
Đang lấy trang 20/56...
  Trang 20: 10 nhân vật
Đang lấy trang 21/56...
  Trang 21: 10 nhân

**DOWNLOAD HTML FROM TXT LINKS**

In [2]:
"""
Bước 2: Đọc character_links.txt, tải HTML từng trang nhân vật, lưu vào folder html_pages/
"""

import requests
import time
import os
import re
from pathlib import Path

INPUT_FILE = "character_links.txt"
OUTPUT_DIR = "html_pages"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

DELAY = 0.5       # Giây nghỉ giữa mỗi request
RETRY = 3         # Số lần thử lại nếu lỗi
TIMEOUT = 15      # Timeout mỗi request


def url_to_filename(url):
    """Chuyển URL thành tên file .html an toàn"""
    # Bỏ schema và domain
    path = re.sub(r"https?://[^/]+", "", url)
    # Thay ký tự không hợp lệ bằng '_'
    name = re.sub(r"[^\w\-.]", "_", path).strip("_")
    # Giới hạn độ dài tên file
    if len(name) > 180:
        name = name[:180]
    return name + ".html"


def download(url, retries=RETRY):
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            resp.encoding = "utf-8"
            if resp.status_code == 200:
                return resp.text
            else:
                print(f"    HTTP {resp.status_code} (lần {attempt})")
        except Exception as e:
            print(f"    Lỗi: {e} (lần {attempt})")
        time.sleep(1)
    return None


def main():
    # Đọc danh sách link
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Không tìm thấy file: {INPUT_FILE}")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]

    total = len(urls)
    print(f"📄 Tổng số link: {total}")

    # Tạo folder output
    Path(OUTPUT_DIR).mkdir(exist_ok=True)

    # Đọc danh sách file đã tải (để resume nếu bị gián đoạn)
    existing = set(os.listdir(OUTPUT_DIR))

    success = 0
    failed = []

    for i, url in enumerate(urls, 1):
        filename = url_to_filename(url)
        filepath = os.path.join(OUTPUT_DIR, filename)

        # Bỏ qua nếu đã tải rồi
        if filename in existing:
            print(f"[{i}/{total}] ⏭  Bỏ qua (đã có): {filename}")
            success += 1
            continue

        print(f"[{i}/{total}] ⬇  {url}")
        html = download(url)

        if html:
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(html)
            success += 1
            print(f"          ✅ Lưu: {filename}")
        else:
            failed.append(url)
            print(f"          ❌ Thất bại: {url}")

        time.sleep(DELAY)

    # Báo cáo
    print(f"\n{'='*50}")
    print(f"✅ Thành công : {success}/{total}")
    print(f"❌ Thất bại   : {len(failed)}/{total}")

    if failed:
        failed_file = "failed_links.txt"
        with open(failed_file, "w", encoding="utf-8") as f:
            f.write("\n".join(failed))
        print(f"\nCác link lỗi đã lưu vào: {failed_file}")
        print("Chạy lại script để tự động retry các link này.")

    print(f"\nHTML đã lưu tại folder: ./{OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

📄 Tổng số link: 553
[1/553] ⬇  https://baotanglichsu.vn/vi/Articles/3098/76235/djong-chi-pham-van-djong-mot-nhan-cach-lon-tron-djoi-vi-nuoc-vi-dan.html
          ✅ Lưu: vi_Articles_3098_76235_djong-chi-pham-van-djong-mot-nhan-cach-lon-tron-djoi-vi-nuoc-vi-dan.html.html
[2/553] ⬇  https://baotanglichsu.vn/vi/Articles/3098/76230/hai-thuong-lan-ong-le-huu-trac-djai-danh-y-cua-dan-toc.html
          ✅ Lưu: vi_Articles_3098_76230_hai-thuong-lan-ong-le-huu-trac-djai-danh-y-cua-dan-toc.html.html
[3/553] ⬇  https://baotanglichsu.vn/vi/Articles/3098/75942/danh-nhan-le-quy-djon-ngoi-sao-sang-tren-bau-troi-van-hoa-viet-nam.html
          ✅ Lưu: vi_Articles_3098_75942_danh-nhan-le-quy-djon-ngoi-sao-sang-tren-bau-troi-van-hoa-viet-nam.html.html
[4/553] ⬇  https://baotanglichsu.vn/vi/Articles/3098/75874/tran-djang-ninh-tam-guong-sang-ve-nguoi-chien-si-cach-mang.html
          ✅ Lưu: vi_Articles_3098_75874_tran-djang-ninh-tam-guong-sang-ve-nguoi-chien-si-cach-mang.html.html
[5/553] ⬇  https://baotang

**HTML TO JSON**

In [3]:
"""
Bước 3: Đọc tất cả file HTML trong folder html_pages/
        Extract nội dung, cấu trúc hóa và lưu ra characters.json
"""

import os
import json
import re
from pathlib import Path
from bs4 import BeautifulSoup

INPUT_DIR  = "html_pages"
OUTPUT_FILE = "characters.json"
BASE_URL   = "https://baotanglichsu.vn"


def filename_to_url(filename):
    """Khôi phục URL xấp xỉ từ tên file (đã encode bởi step2)"""
    name = filename.replace("_html.html", "").replace(".html", "")
    path = name.replace("_", "/", 3)   # 4 segment đầu: vi/Articles/3098/<id>
    # Phần còn lại giữ nguyên dấu _
    return BASE_URL + "/" + name.replace("_", "/", 4).rstrip("/") + ".html"


def parse_html(filepath):
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    # ------------------------------------------------------------------ #
    # Tiêu đề
    # ------------------------------------------------------------------ #
    title = ""
    h2 = soup.find("h2", class_="page-header")
    if h2:
        title = h2.get_text(strip=True)

    # ------------------------------------------------------------------ #
    # Ngày đăng & giờ & lượt xem
    # ------------------------------------------------------------------ #
    date = time_ = views = ""
    stat = soup.find(class_="blogpost-stat")
    if stat:
        text = stat.get_text(" ", strip=True)
        # Ngày: dd/mm/yyyy
        m_date = re.search(r"\d{2}/\d{2}/\d{4}", text)
        if m_date:
            date = m_date.group()
        # Giờ: hh:mm
        m_time = re.search(r"\d{2}:\d{2}", text)
        if m_time:
            time_ = m_time.group()
        # Lượt xem: số nguyên sau icon eye
        m_views = re.search(r"(\d+)\s*$", text.split("đánh")[0].strip())
        if not m_views:
            m_views = re.search(r"eye\S*\s+(\d+)", text)
        # Lấy số đầu tiên đứng sau calendar/clock/eye
        nums = re.findall(r"\b(\d+)\b", text)
        # nums[0]=day part of date, skip date nums; lấy số cuối trước "Điểm"
        views_match = re.search(r"(\d+)\s+Điểm", text)
        if views_match:
            views = views_match.group(1)
        elif len(nums) >= 5:
            views = nums[4]   # sau dd mm yyyy hh mm

    # ------------------------------------------------------------------ #
    # Tóm tắt (abstract)
    # ------------------------------------------------------------------ #
    abstract = ""
    ab_tag = soup.find(class_="article-abstracts")
    if ab_tag:
        abstract = ab_tag.get_text(separator=" ", strip=True)

    # ------------------------------------------------------------------ #
    # Nội dung chi tiết
    # ------------------------------------------------------------------ #
    content_paragraphs = []
    detail = soup.find(class_="article-detail")
    if detail:
        # Xóa các thẻ script/style bên trong
        for tag in detail.find_all(["script", "style"]):
            tag.decompose()

        # Lấy text theo từng đoạn (p, div, h2, h3...)
        for el in detail.find_all(["p", "h2", "h3", "h4", "li", "figcaption"]):
            text = el.get_text(separator=" ", strip=True)
            if text:
                content_paragraphs.append(text)

        # Nếu không có thẻ p/h thì lấy toàn bộ text
        if not content_paragraphs:
            raw = detail.get_text(separator="\n", strip=True)
            content_paragraphs = [l for l in raw.splitlines() if l.strip()]

    content_text = "\n\n".join(content_paragraphs)

    # ------------------------------------------------------------------ #
    # Nguồn
    # ------------------------------------------------------------------ #
    source = ""
    src_tag = soup.find(class_="article-source")
    if src_tag:
        source = src_tag.get_text(strip=True)

    # ------------------------------------------------------------------ #
    # Hình ảnh
    # ------------------------------------------------------------------ #
    images = []
    if detail:
        for img in detail.find_all("img"):
            src = img.get("src", "").strip()
            alt = img.get("alt", "").strip()
            if src:
                if src.startswith("/"):
                    src = BASE_URL + src
                images.append({"url": src, "caption": alt})

    # ------------------------------------------------------------------ #
    # URL gốc (từ og:url hoặc canonical)
    # ------------------------------------------------------------------ #
    url = ""
    og = soup.find("meta", property="og:url")
    if og:
        url = og.get("content", "")
    if not url:
        canon = soup.find("link", rel="canonical")
        if canon:
            url = canon.get("href", "")

    return {
        "title":    title,
        "url":      url,
        "date":     date,
        "time":     time_,
        "views":    views,
        "abstract": abstract,
        "content":  content_text,
        "source":   source,
        "images":   images,
    }


def main():
    if not os.path.isdir(INPUT_DIR):
        print(f"❌ Không tìm thấy folder: {INPUT_DIR}")
        return

    html_files = sorted(Path(INPUT_DIR).glob("*.html"))
    total = len(html_files)
    print(f"📂 Tìm thấy {total} file HTML trong '{INPUT_DIR}/'")

    results = []
    errors  = []

    for i, filepath in enumerate(html_files, 1):
        try:
            data = parse_html(filepath)
            data["_filename"] = filepath.name
            results.append(data)
            status = f"✅ [{i}/{total}] {filepath.name[:60]}"
            if data["title"]:
                status += f"  →  {data['title'][:50]}"
            print(status)
        except Exception as e:
            errors.append({"file": filepath.name, "error": str(e)})
            print(f"❌ [{i}/{total}] {filepath.name} — {e}")

    # Lưu JSON
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n{'='*55}")
    print(f"✅ Đã extract: {len(results)}/{total} nhân vật")
    print(f"❌ Lỗi       : {len(errors)}")
    print(f"📄 Output    : {OUTPUT_FILE}  ({os.path.getsize(OUTPUT_FILE)//1024} KB)")

    if errors:
        with open("extract_errors.json", "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)
        print("⚠️  Chi tiết lỗi lưu vào: extract_errors.json")


if __name__ == "__main__":
    main()


📂 Tìm thấy 553 file HTML trong 'html_pages/'
✅ [1/553] vi_Articles_3098_10287_tin-nguong-tho-cung-hung-vuong-ket-ti  →  Tín ngưỡng thờ cúng Hùng Vương - Kết tinh sức mạnh
✅ [2/553] vi_Articles_3098_12014_liet-si-djau-tien-mai-tang-o-nghia-dj  →  Liệt sĩ đầu tiên mai táng ở Nghĩa địa Phan Bội Châ
✅ [3/553] vi_Articles_3098_12015_co-ba-ma-anh-hung-voi-hai-con-nha-bao  →  Có bà má anh hùng với hai con nhà báo liệt sĩ
✅ [4/553] vi_Articles_3098_12021_khong-gian-ho-chi-minh-o-chau-au.html  →  “Không gian Hồ Chí Minh” ở Châu Âu
✅ [5/553] vi_Articles_3098_12041_phat-hien-hien-vat-cua-nghia-quan-pha  →  Phát hiện hiện vật của nghĩa quân Phan Đình Phùng
✅ [6/553] vi_Articles_3098_12049_cuoc-djoi-nu-tuong-bui-thi-xuan-anh-h  →  Cuộc đời nữ tướng Bùi Thị Xuân: Anh hùng và bi thả
✅ [7/553] vi_Articles_3098_12058_tuyet-chieu-khien-su-thien-trieu-run-  →  Tuyệt chiêu khiến sứ "Thiên triều" run sợ của Lê Đ
✅ [8/553] vi_Articles_3098_12066_nguoi-phu-nu-the-chap-tinh-mang-3djoi  →  Người phụ nữ thế chấ